# 07 - Fine-tune Baltico: que tanto rinde sobre el dataset nuevo

**Que veras aqui (datos REALES):** la evaluacion STANDALONE del modelo U-TAE
**fine-tuneado** al vocabulario del Baltico (EuroCropsML Estonia/Letonia) sobre el
dataset NUEVO. No es una comparacion entre modelos: es **como rinde el modelo
fine-tuneado en el dataset al que se transfirio**.

El modelo parte del backbone U-TAE entrenado en PASTIS (Francia) y se fine-tunea
con una cabeza nueva sobre 18 clases del Baltico:
- **6 clases conservadas** (warm-started desde la cabeza PASTIS: pasture, potatoes,
  winter wheat/barley/rapeseed, spring barley) -- la *bandera* de clases que no se
  olvidan.
- **12 clases nuevas finas** que PASTIS no resuelve (apples, quinces,
  fresh_vegetables, clover, oats, rye...) -- la granularidad enriquecida.

**Indice:** (1) de donde salen los datos, (2) distribucion de clases del test,
(3) accuracy y F1-macro global, (4) rendimiento por clase, (5) matriz de confusion,
(6) predicciones de ejemplo, (7) granularidad fino vs coarse (la hipotesis
papaya/fruits).

In [ ]:
# Parametros papermill.
summary_path = "../../data/transfer/finetune_baltico/finetune_baltico_summary.json"

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display

# Degrada limpio si el summary aun no existe (smoke de CI sin la corrida H100).
summary_file = Path(summary_path)
degraded = not summary_file.is_file()
if degraded:
    print(
        "[degradado] No se encontro", summary_path, "-- ejecuta\n",
        "  python -m scripts.run_finetune_baltico\n",
        "en la H100 para poblar esta libreta con numeros reales.",
    )
    S = {}
else:
    S = json.loads(summary_file.read_text(encoding="utf-8"))
    print("Resumen cargado:", summary_path)

## 1. De donde salen los datos

El test son parcelas REALES de EuroCropsML del pais **target**, cada una con su
serie Sentinel-2 multi-fecha (descargada con textura via Sentinel Hub) y su
etiqueta HCAT. El modelo se entreno en el pais **source** y se evalua en el
**target** (transfer geografico real, no in-distribution).

In [ ]:
if not degraded:
    meta = pl.DataFrame({
        "campo": [
            "Modelo", "Source (entrenamiento)", "Target (test)",
            "Parcelas train", "Parcelas test",
            "Clases (total)", "Clases conservadas", "Clases nuevas",
        ],
        "valor": [
            str(S["model_kind"]), str(S["source"]), str(S["target"]),
            str(S["n_train"]), str(S["n_test"]),
            str(S["n_classes_fine"]), str(S["n_conserved"]), str(S["n_new"]),
        ],
    })
    display(meta)

## 2. Distribucion de clases del test

Cuantas parcelas hay por clase en el conjunto de test. Marca cuales son **nuevas**
(granularidad que PASTIS no tenia) y su soporte -- una clase con muy pocas parcelas
de test da una F1 ruidosa, hay que leerla con su `support`.

In [ ]:
if not degraded:
    pc = pl.DataFrame(S["per_class"]).sort("support", descending=True)
    dist = pc.select(["leaf", "is_new", "support"])
    display(dist)
    fig, ax = plt.subplots(figsize=(9, max(4, 0.4 * pc.height)))
    colors = ["#2ca02c" if nw else "#9aa0a6" for nw in pc["is_new"].to_list()]
    ax.barh(pc["leaf"].to_list(), pc["support"].to_list(), color=colors)
    ax.set_xlabel("Parcelas en el test")
    ax.set_title("Distribucion de clases en el test (verde = clase NUEVA, gris = conservada)")
    ax.invert_yaxis()
    fig.tight_layout()
    plt.show()

## 3. Accuracy y F1-macro global

El rendimiento agregado al nivel **fino** (las 18 clases del Baltico, la prediccion
que el modelo realmente emite) y al nivel **coarse** (colapsado a los grupos PASTIS,
ver seccion 7). La accuracy es el % de parcelas bien clasificadas; la F1-macro
promedia la F1 de cada clase por igual (penaliza fallar en clases raras).

In [ ]:
if not degraded:
    glob = pl.DataFrame({
        "nivel": ["fino (18 clases Baltico)", "coarse (grupos PASTIS)"],
        "accuracy": [S["fine_accuracy"], S["coarse_accuracy"]],
        "f1_macro": [S["fine_macro_f1"], S["coarse_macro_f1"]],
    })
    display(glob)
    print(
        f"Accuracy fino = {S['fine_accuracy']:.1%} | F1-macro fino = {S['fine_macro_f1']:.3f}\n"
        f"Accuracy coarse = {S['coarse_accuracy']:.1%} | F1-macro coarse = {S['coarse_macro_f1']:.3f}"
    )

## 4. Rendimiento por clase

Precision, recall y F1 de cada clase, ordenadas por F1. Asi se ve **sobre que
clases predice bien** el modelo fine-tuneado en el dataset nuevo y cuales le
cuestan. Las clases nuevas (granularidad enriquecida) van marcadas.

In [ ]:
if not degraded:
    tabla = pc.select(["leaf", "is_new", "precision", "recall", "f1", "support"]).sort(
        "f1", descending=True
    )
    display(tabla)
    fig, ax = plt.subplots(figsize=(9, max(4, 0.4 * pc.height)))
    order = pc.sort("f1", descending=False)
    colors = ["#2ca02c" if nw else "#9aa0a6" for nw in order["is_new"].to_list()]
    ax.barh(order["leaf"].to_list(), order["f1"].to_list(), color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel("F1 por clase")
    ax.set_title("F1 por clase del modelo fine-tuneado (verde = clase NUEVA)")
    fig.tight_layout()
    plt.show()

## 5. Matriz de confusion

Que clase predice el modelo cuando la verdad es otra. La diagonal son los aciertos;
fuera de la diagonal estan las confusiones (p. ej. confundir `winter_barley` con
`winter_common_soft_wheat`, fenologicamente parecidas). Normalizada por fila
(recall por clase).

In [ ]:
if not degraded:
    from sklearn.metrics import confusion_matrix

    y_true = S["y_true_leaf"]
    y_pred = S["y_pred_leaf"]
    labels = sorted(set(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("Prediccion")
    ax.set_ylabel("Verdad")
    ax.set_title("Matriz de confusion normalizada por fila (recall por clase)")
    for i in range(len(labels)):
        for j in range(len(labels)):
            if cm_norm[i, j] >= 0.15:
                ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center",
                        fontsize=6, color="white" if cm_norm[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()

## 6. Predicciones de ejemplo

Una muestra de parcelas del test con su verdad y la prediccion del modelo, marcando
aciertos y errores. Da una idea cualitativa de que tan razonable es la salida.

In [ ]:
if not degraded:
    rng = np.random.default_rng(0)
    n = len(y_true)
    take = rng.choice(n, size=min(20, n), replace=False)
    ejemplos = pl.DataFrame({
        "#": [int(i) for i in take],
        "verdad": [y_true[i] for i in take],
        "prediccion": [y_pred[i] for i in take],
        "acierto": ["OK" if y_true[i] == y_pred[i] else "X" for i in take],
    })
    display(ejemplos)
    n_ok = sum(1 for i in take if y_true[i] == y_pred[i])
    print(f"Aciertos en la muestra: {n_ok}/{len(take)}")

## 7. Granularidad: fino vs coarse (la hipotesis papaya/fruits)

El modelo fine-tuneado predice la **hoja fina** (p. ej. `apples`, `quinces`). PASTIS
solo tiene un cubo generico (`Fruits, vegetables, flowers`). Esta seccion muestra la
**granularidad ganada**: cuantas clases NUEVAS finas resuelve el modelo y como se
agrupan en el nivel coarse de PASTIS. La diferencia entre la F1 coarse (mas alta) y
la fina (mas exigente) cuantifica cuanto detalle adicional aporta el modelo sobre lo
que PASTIS solo podia decir.

In [ ]:
if not degraded:
    nuevas = pc.filter(pl.col("is_new")).select(
        ["leaf", "coarse", "f1", "support"]
    ).sort("f1", descending=True)
    print("Clases NUEVAS finas que el modelo resuelve (granularidad sobre PASTIS):")
    display(nuevas)
    resueltas = nuevas.filter(pl.col("f1") > 0.0).height
    print(
        f"\n{resueltas} de {nuevas.height} clases nuevas con F1 > 0 "
        f"(el modelo emite esa granularidad fina).\n"
        f"F1-macro fino {S['fine_macro_f1']:.3f} vs coarse {S['coarse_macro_f1']:.3f}: "
        f"el nivel coarse es mas alto porque agrupa; el fino es la granularidad\n"
        f"extra que PASTIS por si solo NO podia expresar."
    )

## Conclusion

Esta libreta mide, sobre el dataset NUEVO (Baltico), como rinde el U-TAE
fine-tuneado: su accuracy y F1 global, sobre que clases predice bien o mal, sus
confusiones, ejemplos de prediccion, y la granularidad fina que gano respecto al
vocabulario PASTIS del que partio. Los numeros provienen del summary real generado
por `scripts.run_finetune_baltico` en la H100 -- no hay cifras fabricadas; si el
summary no existe la libreta lo indica y degrada limpio.